# **HANDS-ON 1: FUNDAMENTOS DE LLMs CON MODELOS LLAMA**
### Versión de apoyo para facilitación: teoría breve + código comentado

Esta práctica acompaña la sesión **Fundamentos de LLMs y Arquitectura de Llama**.

El objetivo no es entrenar un modelo, sino **ejecutar un modelo Llama ya entrenado**, enviarle un prompt y observar tres aspectos básicos de inferencia:

1. Cómo se ejecuta un modelo localmente con **Ollama**.
2. Cuántos **tokens** procesa el modelo.
3. Cómo cambia la **latencia** al utilizar modelos de distinto tamaño.

>un LLM recibe tokens y genera nuevos tokens de manera secuencial. En esta práctica observaremos ese proceso desde la perspectiva del usuario, sin entrar todavía en técnicas avanzadas.


## **1. CONFIGURACIÓN DEL ENTORNO**

### ¿Qué es Google Colab?
Google Colab es un entorno de notebooks que permite ejecutar código Python en una máquina virtual administrada desde el navegador.

En esta práctica se utiliza una **GPU** porque la inferencia de modelos de lenguaje requiere una gran cantidad de operaciones matemáticas. Una GPU puede ejecutar muchas de estas operaciones en paralelo.

### Componentes que aparecerán
- **GPU T4:** acelerador disponible en algunas sesiones gratuitas de Colab.
- **NVIDIA-SMI:** herramienta de NVIDIA para consultar información de la GPU.
- **Ollama:** software que permite descargar y ejecutar modelos de lenguaje localmente.
- **Servidor local:** proceso que Ollama levanta en la máquina de Colab para recibir solicitudes.
- **Cliente de Python de Ollama:** biblioteca utilizada para enviar mensajes al modelo desde Python.


### **1.1 ACTIVAR Y VERIFICAR LA GPU**

En Colab ve a:

**Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU**

Después ejecuta la siguiente celda.

`nvidia-smi` es una utilidad incluida con los controladores NVIDIA. Aquí únicamente la usamos para verificar que la sesión tiene una GPU disponible y conocer su memoria total.


In [ ]:
# El signo "!" indica a Colab que esta línea se ejecutará como
# un comando del sistema operativo y no como código Python.

# nvidia-smi consulta información de la GPU NVIDIA disponible.
# --query-gpu=name,memory.total indica qué datos queremos mostrar:
#   - name: nombre/modelo de la GPU
#   - memory.total: memoria total de la GPU
# --format=csv muestra el resultado en formato tabular sencillo.
!nvidia-smi --query-gpu=name,memory.total --format=csv


name, memory.total [MiB]
Tesla T4, 15360 MiB


## **2. INSTALACIÓN DE OLLAMA**

### ¿Qué es Ollama?
**Ollama** es una herramienta que facilita la descarga y ejecución de modelos de lenguaje en una computadora o servidor.

En este notebook el modelo se ejecuta **dentro de la máquina virtual de Colab**, por lo que no necesitamos enviar el prompt a una API externa ni utilizar una API key.

Ollama funciona mediante un pequeño **servidor local**. Una vez iniciado, escucha solicitudes en:

`http://localhost:11434`

### Conceptos auxiliares
- **localhost:** nombre que representa a la misma computadora donde se está ejecutando el programa.
- **Puerto 11434:** canal de comunicación que Ollama utiliza por defecto.
- **zstd:** herramienta de compresión que puede ser necesaria durante la instalación y manejo de archivos.
- **curl:** utilidad de línea de comandos utilizada para descargar contenido desde una URL.

> La máquina de Colab es temporal. Si el entorno se reinicia, normalmente hay que volver a instalar Ollama, iniciar el servidor y descargar los modelos.


In [ ]:
# Instalar la utilidad zstd.
# apt-get es el administrador de paquetes del sistema Linux de Colab.
# -qq reduce los mensajes mostrados.
# -y acepta automáticamente la instalación.
!sudo apt-get -qq install -y zstd > /dev/null 2>&1

# Descargar el script oficial de instalación de Ollama y ejecutarlo.
# curl descarga el script desde ollama.com.
# "| sh" envía el contenido descargado al intérprete de comandos.
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

# Mostrar un mensaje cuando termina la instalación.
print("Ollama instalado.")


Ollama instalado.


### **2.1 INICIAR EL SERVIDOR DE OLLAMA**

Instalar Ollama no significa que el servicio esté activo. Debemos iniciar `ollama serve`.

En esta celda aparecen tres bibliotecas:

- **subprocess:** permite iniciar programas del sistema operativo desde Python.
- **time:** permite esperar algunos segundos entre intentos.
- **requests:** permite hacer solicitudes HTTP; aquí se usa para verificar si el servidor de Ollama ya responde.

El servidor se ejecuta **en segundo plano**, permitiendo que el notebook continúe ejecutando otras celdas.


In [ ]:
# subprocess permite iniciar procesos externos desde Python.
import subprocess

# time se utiliza para esperar entre cada intento de conexión.
import time

# requests permite realizar solicitudes HTTP.
import requests

# Iniciar el servidor local de Ollama.
# Popen crea el proceso sin detener la ejecución del notebook.
# stdout y stderr se redirigen para evitar llenar la salida de la celda.
servidor = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Intentar hasta 30 veces conectarse al servidor.
for _ in range(30):
    try:
        # Hacer una petición al servidor local de Ollama.
        requests.get("http://localhost:11434")

        # Si la petición funciona, el servidor ya está listo.
        print("Servidor de Ollama listo.")
        break

    except requests.exceptions.ConnectionError:
        # Si todavía no responde, esperar 1 segundo y volver a intentar.
        time.sleep(1)

else:
    # Este bloque se ejecuta si los 30 intentos fallaron.
    print("El servidor no respondió. Vuelve a ejecutar esta celda.")

# Mostrar la versión instalada de Ollama.
!ollama --version


Servidor de Ollama listo.
ollama version is 0.40.3


## **3. DESCARGA DE LOS MODELOS LLAMA**

### ¿Qué significa 3B u 8B?
La letra **B** significa *billions* (miles de millones) y se refiere aproximadamente al número de parámetros del modelo.

En esta práctica se comparan:

- `llama3.2:3b` → modelo de aproximadamente 3 mil millones de parámetros.
- `llama3.1:8b` → modelo de aproximadamente 8 mil millones de parámetros.

### ¿Qué es un parámetro?
Un **parámetro** es un valor numérico aprendido durante el entrenamiento del modelo. Millones o miles de millones de estos valores determinan cómo transforma una entrada en una predicción.

Un modelo más grande puede tener mayor capacidad, pero también suele requerir más memoria y cómputo.

### ¿Qué hace `ollama pull`?
Descarga a la máquina local los archivos necesarios para ejecutar un modelo.

La etiqueta del modelo debe coincidir exactamente con la utilizada posteriormente en Python.


In [ ]:
# Guardar los identificadores de los modelos en variables.
# Esto evita repetir manualmente los nombres y reduce errores de escritura.
MODELO_LIGERO = "llama3.2:3b"
MODELO_GRANDE = "llama3.1:8b"

# Descargar el modelo ligero.
!ollama pull {MODELO_LIGERO}

# Descargar el modelo grande.
!ollama pull {MODELO_GRANDE}

# Mostrar los modelos disponibles actualmente en Ollama.
!ollama list




NAME           ID              SIZE      MODIFIED               
llama3.1:8b    46e0c10c039e    4.9 GB    Less than a second ago    
llama3.2:3b    a80c4f17acd5    2.0 GB    54 seconds ago            


## **4. CLIENTE DE PYTHON PARA OLLAMA**

Hasta ahora Ollama está instalado y su servidor está funcionando. Ahora necesitamos una forma cómoda de comunicarnos con él desde Python.

La biblioteca `ollama` funciona como **cliente**: nuestro código Python crea una solicitud, la envía al servidor local y recibe la respuesta generada por el modelo.

### Cliente y servidor
- **Servidor:** ejecuta el modelo y atiende solicitudes.
- **Cliente:** envía las solicitudes al servidor.

La dirección `http://localhost:11434` indica que ambos están ejecutándose dentro de la misma máquina virtual de Colab.


In [ ]:
# Instalar la biblioteca de Python que permite comunicarnos con Ollama.
!pip install ollama -q

# Importar la biblioteca.
import ollama

# Importar time porque más adelante mediremos tiempos de ejecución.
import time

# Crear un cliente y especificar dónde está el servidor de Ollama.
client = ollama.Client(host="http://localhost:11434")

# Confirmar que la variable del cliente fue creada.
print("Cliente de Ollama inicializado correctamente.")


Cliente de Ollama inicializado correctamente.


## **5. DE PALABRAS A TOKENS**

### ¿Qué es un token?
Un **token** es una unidad en la que el modelo divide el texto antes de procesarlo. Un token puede corresponder a una palabra, una parte de una palabra, un signo o algún otro fragmento.

El modelo no procesa directamente las letras como nosotros las vemos. El texto se convierte primero en tokens y después en representaciones numéricas.

### ¿Qué es inferencia?
La **inferencia** es el proceso de utilizar un modelo ya entrenado para producir una respuesta ante una nueva entrada.

En esta práctica no estamos entrenando Llama: estamos haciendo inferencia.


### **5.1 PRIMERA LLAMADA A LLAMA**

El notebook original denomina esta llamada **zero-shot**. Aquí solamente necesitamos entenderlo como:

> pedirle al modelo que realice una tarea **sin darle ejemplos previos**.

No profundizaremos todavía en técnicas de prompt engineering; únicamente usamos el término porque aparece en la práctica.

La conversación se representa con una lista de mensajes. Cada mensaje contiene:

- `role`: quién envía el mensaje.
- `content`: el texto del mensaje.

En este caso el rol es `user`, porque el mensaje representa la pregunta del usuario.


In [ ]:
# Guardar la pregunta que enviaremos al modelo.
prompt = "¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?"

# Enviar el mensaje al modelo ligero.
response = client.chat(
    # Indicar qué modelo debe utilizar Ollama.
    model=MODELO_LIGERO,

    # messages representa la conversación.
    # Cada elemento tiene:
    #   role -> quién habla
    #   content -> qué texto envía
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

# Mostrar primero la pregunta original.
print(prompt)

# Mostrar únicamente el texto que generó el modelo.
print(response.message.content)


¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?
**Diferencia entre RAM y almacenamiento en una computadora**

La RAM (Random Access Memory) y el almacenamiento en una computadora son dos conceptos fundamentales que trabajan juntos para proporcionar espacio para que la computadora almacene y procese información.

**RAM (Random Access Memory)**

La RAM es una parte de la computadora que permite al sistema operativo y las aplicaciones acceder rápidamente a la información almacenada en la memoria. La RAM es un tipo de memoria volátil, lo que significa que la información se pierde cuando la computadora se apaga.

La RAM es utilizada para:

* Almacenar temporales datos y procesos
* Procesar información en tiempo real
* Encontrar rápido acceso a información reciente

**Almacenamiento**

El almacenamiento, por otro lado, se refiere a la capacidad de la computadora para almacenar datos a largo plazo, incluso cuando la computadora está apagada. El almacenamiento puede

### **5.2 INSPECCIONAR LA RESPUESTA COMPLETA**

El modelo no devuelve únicamente texto. Ollama también incluye **metadatos** sobre la ejecución, por ejemplo:

- nombre del modelo,
- duración,
- cantidad de tokens procesados,
- cantidad de tokens generados.

`model_dump_json()` convierte el objeto de respuesta en una representación JSON legible para poder inspeccionarlo.


In [ ]:
# Convertir el objeto completo de respuesta a JSON.
# indent=2 agrega sangría para que sea más fácil de leer.
print(response.model_dump_json(indent=2))


{
  "model": "llama3.2:3b",
  "created_at": "2026-10-12T22:31:44.625713769Z",
  "done": true,
  "done_reason": "stop",
  "total_duration": 123360145880,
  "load_duration": 64052923899,
  "prompt_eval_count": 43,
  "prompt_eval_duration": 51557667000,
  "eval_count": 577,
  "eval_duration": 7744089000,
  "message": {
    "role": "assistant",
    "content": "**Diferencia entre RAM y almacenamiento en una computadora**\n\nLa RAM (Random Access Memory) y el almacenamiento en una computadora son dos conceptos fundamentales que trabajan juntos para proporcionar espacio para que la computadora almacene y procese información.\n\n**RAM (Random Access Memory)**\n\nLa RAM es una parte de la computadora que permite al sistema operativo y las aplicaciones acceder rápidamente a la información almacenada en la memoria. La RAM es un tipo de memoria volátil, lo que significa que la información se pierde cuando la computadora se apaga.\n\nLa RAM es utilizada para:\n\n* Almacenar temporales datos y proce

## **6. CONTEO DE TOKENS**

Ollama reporta dos valores especialmente útiles:

- `prompt_eval_count`: número de tokens de la entrada.
- `eval_count`: número de tokens generados en la salida.

Estos valores permiten observar cuánto texto procesó y produjo el modelo.

En esta práctica los tokens **no representan un costo monetario**, porque el modelo está corriendo en la propia sesión de Colab. Aun así, siguen siendo importantes porque afectan el trabajo computacional y el uso de la ventana de contexto.


In [ ]:
# Obtener la cantidad de tokens que tenía el prompt.
tokens_prompt = response.prompt_eval_count

# Obtener la cantidad de tokens generados como respuesta.
tokens_respuesta = response.eval_count

# Mostrar ambas cantidades.
print("Tokens del prompt:", tokens_prompt)
print("Tokens de la respuesta:", tokens_respuesta)

# Sumar entrada y salida para obtener los tokens procesados en esta interacción.
print("Tokens totales:", tokens_prompt + tokens_respuesta)


Tokens del prompt: 43
Tokens de la respuesta: 577
Tokens totales: 620


## **7. LATENCIA DE INFERENCIA**

### ¿Qué es latencia?
La **latencia** es el tiempo que transcurre entre enviar una solicitud y obtener la respuesta.

Para medirla utilizamos:

- `time.time()` antes de llamar al modelo.
- `time.time()` después de recibir la respuesta.
- La diferencia entre ambos valores corresponde al tiempo transcurrido.

Esta medición incluye el tiempo total percibido por nuestro código durante la llamada.


In [ ]:
# Registrar el tiempo justo antes de enviar la solicitud.
inicio = time.time()

# Realizar nuevamente la inferencia con el mismo modelo y prompt.
response_tiempo = client.chat(
    model=MODELO_LIGERO,
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

# Calcular cuántos segundos transcurrieron.
duracion = time.time() - inicio

# Mostrar el tiempo con dos decimales.
print(f"Tiempo de respuesta: {duracion:.2f} segundos")


Tiempo de respuesta: 9.07 segundos


## **8. COMPARACIÓN DE DOS TAMAÑOS DE MODELO**

Ahora se repite la misma tarea con el modelo de 8B.

La comparación busca observar que elegir un modelo es una **decisión de ingeniería**. Debemos considerar al menos:

- calidad de la respuesta,
- tiempo de respuesta,
- memoria disponible,
- tamaño del modelo.

### ¿Qué es `load_duration`?
Cuando un modelo se usa por primera vez, Ollama puede necesitar cargarlo en memoria. Ese proceso añade tiempo adicional.

Ollama reporta `load_duration` en **nanosegundos**. Por eso el código divide entre `1e9` para convertir el valor a segundos.

> Para una comparación más justa de inferencia, conviene distinguir el tiempo de carga inicial del tiempo de generación.


In [ ]:
# Registrar el instante antes de llamar al modelo grande.
inicio = time.time()

# Enviar exactamente el mismo prompt al modelo grande.
response_grande = client.chat(
    model=MODELO_GRANDE,
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

# Calcular el tiempo total de la llamada.
duracion_grande = time.time() - inicio

# Ollama reporta load_duration en nanosegundos.
# Dividir entre 1e9 lo convierte a segundos.
carga_grande = response_grande.load_duration / 1e9

# Calcular tokens totales de la respuesta original del modelo ligero.
tokens_ligero = response.prompt_eval_count + response.eval_count

# Calcular tokens totales del modelo grande.
tokens_grande = (
    response_grande.prompt_eval_count
    + response_grande.eval_count
)

# Mostrar datos del modelo ligero.
print(
    f"Modelo ligero ({MODELO_LIGERO}): "
    f"{duracion:.2f} s — {tokens_ligero} tokens"
)

# Mostrar datos del modelo grande e indicar cuánto tiempo correspondió a carga.
print(
    f"Modelo grande ({MODELO_GRANDE}): "
    f"{duracion_grande:.2f} s — {tokens_grande} tokens "
    f"(de los cuales {carga_grande:.2f} s fueron carga del modelo)"
)

# Mostrar el texto generado por el modelo grande.
print(
    "\nRespuesta del modelo grande:\n",
    response_grande.message.content
)


Modelo ligero (llama3.2:3b): 9.07 s — 620 tokens
Modelo grande (llama3.1:8b): 19.59 s — 74 tokens (de los cuales 17.70 s fueron carga del modelo)

Respuesta del modelo grande:
 La RAM (Memoria de Acceso Aleatorio) se utiliza para almacenar datos temporarios y datos del sistema en memoria mientras que el almacenamiento se utiliza para almacenar datos permanentes en la unidad de disco duro.


# **9. CIERRE DE LA PRÁCTICA**

Al finalizar este hands-on debes poder explicar:

1. Qué es un **LLM** y qué significa realizar inferencia.
2. Qué es un **token**.
3. Qué es un **parámetro** y qué representan tamaños como 3B y 8B.
4. Qué es **Ollama** y por qué permite ejecutar Llama dentro de Colab.
5. La diferencia entre un **cliente** y un **servidor**.
6. Qué es `localhost` y por qué Ollama utiliza el puerto `11434`.
7. Cómo enviar un mensaje a Llama mediante `client.chat()`.
8. Qué representan `prompt_eval_count` y `eval_count`.
9. Qué significa **latencia de inferencia**.
10. Por qué un modelo más grande no siempre es automáticamente la mejor opción.

### Pregunta de reflexión
Si el modelo de 3B responde correctamente y con menor latencia que el de 8B para una tarea determinada, ¿qué ventaja tendría utilizar el modelo más grande?

La respuesta debe considerar que la selección del modelo depende del caso de uso y de los recursos disponibles.


**Challange_1**

Celda 1 — Definir la lista de preguntas Una lista de Python permite almacenar varias preguntas en una sola variable. Aquí usaremos preguntas frecuentes de soporte técnico

In [ ]:
# Definir la lista de preguntas de soporte técnico
preguntas = [
    "¿Qué puedo hacer si mi computadora está muy lenta?",
    "¿Cómo puedo recuperar mi contraseña si la olvidé?"
]

# Mostrar las preguntas disponibles
for i, pregunta in enumerate(preguntas, start=1):
    print(f"Pregunta {i}: {pregunta}")

Pregunta 1: ¿Qué puedo hacer si mi computadora está muy lenta?
Pregunta 2: ¿Cómo puedo recuperar mi contraseña si la olvidé?


Celda 2 — Consultar la pregunta 1 y guardar resultado_1 Enviaremos la misma pregunta a ambos modelos para comparar respuesta, tiempo y tokens.

In [ ]:
# Seleccionar la primera pregunta
pregunta_1 = preguntas[0]

# ----- Modelo ligero -----
inicio = time.time()

respuesta_ligero_1 = client.chat(
    model=MODELO_LIGERO,
    messages=[
        {
            "role": "user",
            "content": pregunta_1
        }
    ]
)

# Calcular tiempo de respuesta
tiempo_ligero_1 = time.time() - inicio

# Calcular tokens totales
tokens_ligero_1 = (
    respuesta_ligero_1.prompt_eval_count
    + respuesta_ligero_1.eval_count
)

# Guardar los datos de la pregunta 1
resultado_1 = {
    "pregunta": pregunta_1,
    "respuesta_ligero": respuesta_ligero_1.message.content,
    "tiempo_ligero": tiempo_ligero_1,
    "tokens_ligero": tokens_ligero_1
}

# ----- Mostrar el resultado de forma ordenada -----

print("=" * 80)
print("PREGUNTA 1")
print("=" * 80)

print(f"\nPregunta:")
print(resultado_1["pregunta"])

print("\n" + "-" * 80)
print(f"MODELO: {MODELO_LIGERO}")
print("-" * 80)

print("\nRespuesta:")
print(resultado_1["respuesta_ligero"])

print("\n" + "-" * 80)
print("MÉTRICAS")
print("-" * 80)

print(f"Tiempo de respuesta: {resultado_1['tiempo_ligero']:.2f} segundos")
print(f"Tokens totales: {resultado_1['tokens_ligero']}")

PREGUNTA 1

Pregunta:
¿Qué puedo hacer si mi computadora está muy lenta?

--------------------------------------------------------------------------------
MODELO: llama3.2:3b
--------------------------------------------------------------------------------

Respuesta:
**Soluciones para mejorar el rendimiento de tu computadora**

Si tu computadora está muy lenta, hay varias soluciones que puedes intentar para mejorar su rendimiento. Aquí te presento algunas opciones:

### 1. **Reinicia tu computadora**

A veces, simplemente reiniciar tu computadora puede hacer maravillas. Cierra todas las aplicaciones y espera a que se detengan, luego reinicia tu computadora y observa si el rendimiento mejora.

### 2. **Baja la cantidad de aplicaciones abiertas**

Las aplicaciones en segundo plano pueden consumir recursos del sistema y ralentizar tu computadora. Cierra todas las aplicaciones que no estés utilizando y reduce la cantidad de aplicaciones abiertas.

### 3. **Instala actualizaciones de softwa

Celda 4 — Juntar los resultados Agrupamos resultado_1 y resultado_2 en una lista llamada resultados.

In [ ]:
# Crear la lista resultados
resultados = []

# Agregar los dos diccionarios
resultados.append(resultado_1)
#resultados.append(resultado_2)

print(f"Resultados almacenados: {len(resultados)}")

Resultados almacenados: 1


5. Muestra la comparación: Imprime resultados y concluye si el modelo ligero resolvió las 3 preguntas satisfactoriamente.

In [ ]:
import pandas as pd

# Crear la lista con los dos resultados
resultados = [resultado_1] #resultados = [resultado_1, resultado_2]

# Preparar los datos para la tabla
comparacion = []

for resultado in resultados:
    comparacion.append({
        "Pregunta": resultado["pregunta"],
        "Tiempo (s)": round(resultado["tiempo_ligero"], 2),
        "Tokens": resultado["tokens_ligero"]
    })

# Crear y mostrar la tabla
tabla_resultados = pd.DataFrame(comparacion)

display(tabla_resultados)

,Pregunta,Tiempo (s),Tokens
0,¿Qué puedo hacer si mi computadora está muy le...,14.29,679
